In [1]:
from pyspark.sql import SparkSession


In [2]:
spark = SparkSession.builder \
    .appName("RDD_OPs") \
    .getOrCreate()

26/06/03 16:12:14 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
!hadoop fs -ls /user/tanuj/1MB


Found 5 items
-rw-r--r--   2 tanujkumawat3008 hadoop    1058478 2026-06-03 15:34 /user/tanuj/1MB/customers.csv
-rw-r--r--   2 tanujkumawat3008 hadoop     837537 2026-06-03 15:34 /user/tanuj/1MB/items.csv
-rw-r--r--   2 tanujkumawat3008 hadoop     864432 2026-06-03 15:34 /user/tanuj/1MB/orders.csv
-rw-r--r--   2 tanujkumawat3008 hadoop     855780 2026-06-03 15:34 /user/tanuj/1MB/payments.csv
-rw-r--r--   2 tanujkumawat3008 hadoop     772763 2026-06-03 15:34 /user/tanuj/1MB/shippings.csv


In [4]:
hdfs_path = "/user/tanuj/1MB/customers.csv"
rdd = spark.sparkContext.textFile(hdfs_path)

In [9]:
header = rdd.first()


In [10]:
no_header_rdd = rdd.filter(lambda row: row!=header).map(lambda row: row.split(','))

In [11]:
no_header_rdd.first()

['0', 'Customer_0', 'Pune', 'Maharashtra', 'India', '2023-05-17', 'True']

In [12]:
reduced_rdd = no_header_rdd.map(lambda row:(row[2],1)).reduceByKey(lambda a,b:a+b)

In [16]:
reduced_rdd.collect()

[('Pune', 2216),
 ('Hyderabad', 2194),
 ('Mumbai', 2093),
 ('Delhi', 2161),
 ('Bangalore', 2175),
 ('Ahmedabad', 2157),
 ('Chennai', 2151),
 ('Kolkata', 2185)]

In [17]:
grouped_rdd = no_header_rdd.map(lambda row: (row[2],1)).groupByKey()

In [21]:
grouped_rdd.collect()

[('Pune', <pyspark.resultiterable.ResultIterable at 0x7fd139967050>),
 ('Hyderabad', <pyspark.resultiterable.ResultIterable at 0x7fd13997d2d0>),
 ('Mumbai', <pyspark.resultiterable.ResultIterable at 0x7fd13997db50>),
 ('Delhi', <pyspark.resultiterable.ResultIterable at 0x7fd13996e250>),
 ('Bangalore', <pyspark.resultiterable.ResultIterable at 0x7fd13997e4d0>),
 ('Ahmedabad', <pyspark.resultiterable.ResultIterable at 0x7fd13997e350>),
 ('Chennai', <pyspark.resultiterable.ResultIterable at 0x7fd13997e110>),
 ('Kolkata', <pyspark.resultiterable.ResultIterable at 0x7fd13997dcd0>)]

In [19]:
group_result_rdd = grouped_rdd.map(lambda row:(row[0],len(row[1])))

In [20]:
group_result_rdd.collect()

[('Pune', 2216),
 ('Hyderabad', 2194),
 ('Mumbai', 2093),
 ('Delhi', 2161),
 ('Bangalore', 2175),
 ('Ahmedabad', 2157),
 ('Chennai', 2151),
 ('Kolkata', 2185)]

# Repartition

In [22]:
rdd.getNumPartitions()

2

In [23]:
rddPartitionsMore = rdd.repartition(4)

In [24]:
rddPartitionsMore.getNumPartitions()

4

In [26]:
rddPartitionsLess = rdd.repartition(1)

In [27]:
rddPartitionsLess.getNumPartitions()

1

# coalesce
    coalesce() is used to reduce the number of partitions in an RDD or DataFrame without performing a full shuffle (by default).
   ### cannot increase just decrease

In [30]:
rdd.getNumPartitions()

2

In [28]:
rdd.coalesce(1).getNumPartitions()

1

In [31]:
rdd.coalesce(4).getNumPartitions() #didnt increased nor gave any error

2